# Sélection maladies Orphanet

Outil d'annotation manuelle pour les patients dont le gène est associé à plusieurs maladies.


## Préparation


In [ ]:
import pandas as pd
from IPython.display import clear_output
import re
import ast


def nettoyer_html(texte):
    texte = str(texte)
    texte = re.sub(r'<[^>]+>', ' ', texte)      # supprime les balises
    texte = texte.replace('&nbsp;', ' ')         # entités HTML courantes
    texte = texte.replace('&amp;', '&')
    texte = texte.replace('&lt;', '<')
    texte = texte.replace('&gt;', '>')
    texte = re.sub(r'\s+', ' ', texte)           # multiple espaces/\n ? un seul espace
    return texte.strip()

# Dataset contenant les comptes rendus
crData = pd.read_csv("../../../data/raw_data/cr_genetique_exomes_with_text.csv", sep=";")
colonnes_cr = ["DOCUMENT_ID","PATIENT_ID","TEXT"]

INPUT_FILE  = "patients_multi_diseases.csv"
OUTPUT_FILE = "choice_output.csv"

df = pd.read_csv(INPUT_FILE)
all_ids = list(df["ID_PAT_ETUDE"].unique())

try:
    done_ids = set(pd.read_csv(OUTPUT_FILE)["ID_PAT_ETUDE"].unique())
except FileNotFoundError:
    done_ids = set()

all_ids = sorted(list(df["ID_PAT_ETUDE"].unique()))
remaining = [p for p in all_ids if p not in done_ids]

print("Total patients :", len(all_ids))
print("Deja annotes   :", len(done_ids))
print("Restants       :", len(remaining))
if done_ids:
    print("Reprise au patient", len(done_ids) + 1)

## Annotation

Lancez la cellule ci-dessous. Pour chaque patient : 
- tapez le **numéro** de la maladie choisie et appuyez sur **Entrée**
- taper `s` + Entrée pour passer un patient sans l'annoter.
- taper `q` + Entrée pour arrêter et sauvegarder la progression.
- taper `n` + Entrée pour lire le compte rendu suivant si disponible.
- taper `p` + Entrée pour lire le compte précédent. 



In [ ]:
def save_result(pat_id, disease_name, orpha_code):
    new_row = pd.DataFrame([{
        "ID_PAT_ETUDE"     : pat_id,
        "real_disease_name": disease_name,
        "real_disease_code": orpha_code,
    }])
    try:
        current = pd.read_csv(OUTPUT_FILE)
        updated = pd.concat([current, new_row]).drop_duplicates("ID_PAT_ETUDE", keep="last")
    except FileNotFoundError:
        updated = new_row
    updated.to_csv(OUTPUT_FILE, index=False)


SEP = "-" * 60

remaining = sorted(remaining, reverse=False)

for i, pat_id in enumerate(remaining):
    pat_data = df[df["ID_PAT_ETUDE"] == pat_id]

    gene    = ", ".join(pat_data["gene_symbol"].unique())
    ipp     = pat_data["IPP_clef"].iloc[0]
    gene_id = ", ".join(pat_data["GeneId"].unique().astype(str))
    hpo     = (
        pat_data["hpo_unique_list_name"].iloc[0]
        if "hpo_unique_list_name" in pat_data.columns and pd.notna(pat_data["hpo_unique_list_name"].iloc[0])
        else "—"
    )

    choices = pat_data[["OrphaCode", "DiseaseName"]].drop_duplicates().reset_index(drop=True)

    # Affichage des comptes rendus
    crs_patient = crData[crData["PATIENT_ID"] == pat_id][["DOCUMENT_ID", "TEXT"]].reset_index(drop=True)
    textes_cr   = [nettoyer_html(t) for t in crs_patient["TEXT"].tolist()]
    doc_ids     = crs_patient["DOCUMENT_ID"].tolist()

    cr_i = 0
    while True:
        clear_output(wait=True)
        print(SEP)
        print("  Patient", i + 1, "/", len(remaining))
        print(SEP)
        print("  Patient ID :", pat_id)
        print("  IPP_clef   :", ipp)
        print("  Gene       :", gene)
        print("  Gene ID    :", gene_id)
        print("  Termes HPO :", hpo)
        print()
        if textes_cr:
            print(f"  --- Compte rendu {cr_i + 1} / {len(textes_cr)} --- Document ID : {doc_ids[cr_i]}")
            print(textes_cr[cr_i])
        else:
            print("  (aucun compte rendu)")
        print()
        print("  Maladies proposees :")
        for j, row in choices.iterrows():
            print("   ", j+1, "-", row["DiseaseName"], " (ORPHA:" + str(row["OrphaCode"]) + ")")
        print()
        print("  [n] CR suivant   [p] CR precedent   [s] Passer   [q] Quitter   ou un numero")
        print(SEP)

        rep = input("  Votre choix : ").strip().lower()

        #Affichage CR
        print()
        

        if rep == "n" and textes_cr and cr_i < len(textes_cr) - 1:
            cr_i += 1
        elif rep == "p" and cr_i > 0:
            cr_i -= 1
        elif rep == "q":
            print("Annotation arretee. Relancez pour reprendre.")
            break
        elif rep == "s":
            print("  -> Patient", pat_id, "passe.")
            break
        elif rep.isdigit() and 1 <= int(rep) <= len(choices):
            idx_choice = int(rep) - 1
            row = choices.iloc[idx_choice]
            save_result(pat_id, row["DiseaseName"], row["OrphaCode"])
            print("  OK Sauvegarde :", row["DiseaseName"])
            break

    if rep == "q":
        break

else:
    clear_output(wait=True)
    print("Tous les patients ont ete annotes !")
    print("Fichier :", OUTPUT_FILE)


## Export final

Relancez la cellule ci-dessous pour joindre les nouvelles annotations au fichier d'entrée.

In [ ]:
annotations = pd.read_csv(OUTPUT_FILE)

drop_cols = [c for c in ["OrphaCode", "DiseaseName", "GeneSymbol", "GeneId"] if c in df.columns]
df_patients = df.drop_duplicates("ID_PAT_ETUDE")[[c for c in df.columns if c not in drop_cols]]

df_export = df_patients.merge(annotations, on="ID_PAT_ETUDE", how="left")
df_export.to_csv("patients_one_disease.csv", index=False)

annotated = df_export["real_disease_code"].notna().sum()
print("Export : patients_one_disease.csv")
print("Annotes :", annotated, "/", len(df_export))
df_export.head()